# 4.2 The long tail, on taxi fares

[04.1](04.1-families.ipynb) drew six families from known parameters and named the mechanism
behind each. This notebook puts one of them on real data. The question is not "what does a
taxi fare look like" — it is *which family, and why*, because the answer decides what counts
as an expensive ride. It assumes 04.1's mechanisms, the central limit theorem in particular
(a product becomes a sum under the log), and the goad pieces 04.1 introduced:
`DistributionFitter`, `fit_table` with its two winners, `PlotFits` and `QQPlot`.

`taxis`: 6,433 NYC rides. Think about the mechanism first. A fare is a base charge plus a
meter that runs on distance *and* on time stuck in traffic, plus tolls and surcharges on some
rides. Distance itself is skewed — most rides are short, a few cross the city — and slow
traffic multiplies the cost of every extra kilometre. Several factors, mostly *multiplying*
each other, with a hard floor at the base charge and no ceiling: that is the product
mechanism from 04.1 §4.1.2, and it predicts a lognormal. Check the prediction in three steps:
the summary that betrays the tail, the picture with and without the log, and then the fitter.

In [ ]:
import numpy as np

from goad_toolkit.analytics import DistributionFitter, fit_table
from goad_toolkit.distributions import DistributionRegistry
from goad_toolkit.visualizer import FitPlotSettings, HistogramPlot, PlotFits, PlotSettings, QQPlot

from wa_analyzer.data import load_showcase

## 4.2.1 The summary that betrays the tail

In [ ]:
taxis = load_showcase("taxis")
fare = taxis["fare"].to_numpy()

print(f"mean:   {fare.mean():.2f}")
print(f"median: {np.median(fare):.2f}")

The mean sits well above the median — the tail pulls it up. For a symmetric distribution the
two would match, so this gap is the first sign that "mean ± std" would describe a fare that
does not exist. Now the picture, twice: raw, and after taking the log. **The log is the
fix**: if the mechanism is multiplicative, `log(fare)` is a *sum* of the logged factors, and
04.1 §4.1.2 says a sum should look normal.

In [ ]:
tail = PlotSettings(
    figsize=(10, 3.5),  # ty: ignore[invalid-argument-type]
    title="The same fares, before and after the log",
    subplot_titles=["fare: a long right tail", "log(fare): the tail becomes a bell"],
    max_cols=2,
)
host = HistogramPlot(tail)
fig, axes = host.create_figure(n_plots=2)
host.plot_on_axes(HistogramPlot(tail), axes[0], data=fare, color="steelblue")
_ = host.plot_on_axes(HistogramPlot(tail), axes[1], data=np.log(fare), color="steelblue")

## 4.2.2 Let the fitter say it

The picture supports the prediction; the fitter makes it a number. `fit()` on the raw fares
tries every continuous family in the registry and ranks them. Read the table for two things:
does `lognorm` win, and by how much does it beat `norm` — the family a z-score rule quietly
assumes.

In [ ]:
fitter = DistributionFitter(DistributionRegistry(), seed=42)
fare_fits = fitter.fit(fare, discrete=False)
fit_table(fare_fits)[["distribution", "log_likelihood", "ks_pvalue", "best_likelihood", "best_ks"]]

In [ ]:
fig = PlotFits(PlotSettings(figsize=(12, 4), xlabel="fare", ylabel="density", title="Taxi fares: the top three fits")).plot(
    data=fare, fit_results=fare_fits, fitplotsettings=FitPlotSettings(bins=40, max_fits=3),
)

## 4.2.3 The tail, where families differ

A histogram with a curve on it is dominated by the bulk of the data, which is exactly where
families look most alike. The tail is where they differ, and the tail is what a `QQPlot`
shows: sorted fares against the quantiles a fitted family predicts, with a reference line.
Points that bend away from the line are values the family cannot explain.

In [ ]:
norm_fit = fitter.fit_distribution("norm", fare)
lognorm_fit = fitter.fit_distribution("lognorm", fare)

qq = PlotSettings(
    figsize=(10, 4.5),  # ty: ignore[invalid-argument-type]
    title="Which family explains the expensive rides?",
    subplot_titles=["fare vs. fitted normal", "fare vs. fitted lognormal"],
    xlabel="theoretical quantile",
    ylabel="fare",
    max_cols=2,
)
host = QQPlot(qq)
fig, axes = host.create_figure(n_plots=2)
host.plot_on_axes(QQPlot(qq), axes[0], data=fare, distribution=norm_fit.frozen_dist)  # ty: ignore[unresolved-attribute]
_ = host.plot_on_axes(QQPlot(qq), axes[1], data=fare, distribution=lognorm_fit.frozen_dist)  # ty: ignore[unresolved-attribute]

The normal qq-plot bends sharply away from the line in the upper tail — exactly where a
histogram is least readable, and exactly where a "z-score above 3" rule would flag ordinary
expensive rides as anomalies. The lognormal tracks the line much further out. Same data;
the difference is what each family expects the tail to look like.

## 4.2.4 The most expensive ride

What that means for the single most expensive ride in the set:

In [ ]:
worst_fare = fare.max()
z = (worst_fare - fare.mean()) / fare.std()
tail_prob = 1 - lognorm_fit.frozen_dist.cdf(worst_fare)  # ty: ignore[unresolved-attribute]
print(f"most expensive fare: ${worst_fare:.2f}, n={len(fare):,} rides")
print(f"z-score under a normal fit: {z:.1f}  (a real normal puts this at ~1-in-10^32)")
print(f"P(fare >= this) under the fitted lognormal: {tail_prob:.1e}  (~1-in-{1/tail_prob:,.0f})")

Still rare under the lognormal fit — 1-in-19,000 is not "expected" — but the two models
disagree by about 27 orders of magnitude on *how* rare. The normal model calls this ride
essentially impossible; the lognormal model calls it an unlucky but real draw from a tail
every long ride is exposed to. Neither model says "delete this row"; only the wrong one says
"this cannot have happened." That is reason 2 from the top of [04.1](04.1-families.ipynb): an outlier is a
claim about a distribution, and the claim is only as good as the family behind it.

---

**Where this goes next.** Here the fitter and the mechanism agreed: one process, one family,
one tail. [04.3-penguins](04.3-penguins.ipynb) is the case where the fitter cannot agree with
anything — every family rejected — and the reason is not a strange family but a population
that is really several groups. The tool is the same; what changes is the question you ask
when the fit fails.